In [40]:
# Practica 8 Realizado por Angel Antolin

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression

# ----------------- EJERCICIO 1 ---------------------------

Apartado a) y b) creacion del objeto y metodo __init__

In [41]:
# Declaro la clase y en el metodo init se guarda el numero de clases, la lista de LinearRegression y el OneHotEncoder
class rlm(object):

    def __init__(self, n_clases):

        self.n_clases = n_clases

        self.regresores = []
        
        for _ in range(self.n_clases):

            self.regresores.append(LinearRegression())

        self.encoder = OneHotEncoder(sparse_output = False)

# Apartado c) metodo fit para ajustar parametros de los regresores

    def fit(self, X, Y):

        # El encoder requiere un formato de matrices 2D de la forma [[2], [1], [0]]
        y_matriz2d = np.array(Y).reshape(-1,1)

        y_onehot = self.encoder.fit_transform(y_matriz2d)

        # El numero de columnas es el numero de clases encontro el codificador, si no coincide con el del init, error
        n_clases_halladas = y_onehot.shape[1]

        if(n_clases_halladas != self.n_clases):

            raise ValueError(
                
                f"Error: El init definió {self.n_clases} clases, "
                f"pero se han detectado {n_clases_halladas} clases en los datos."
            )

        # Si no hay errores, entrenar los regresores
        for i in range(self.n_clases):

            salida_clase_i = y_onehot[:, i]
            self.regresores[i].fit(X, salida_clase_i)

# Apartado d) metodo predict devuelve el array de enteros concordantes con fit

    def predict(self, test):

        n_muestras = test.shape[0]

        #Matriz de 0, filas son las muestras, columnas las clases
        salidas = np.zeros((n_muestras, self.n_clases))

        #Guardar la prediccion de cada regresor sobre cada muestra
        for i in range(self.n_clases):

            salidas[:, i] = self.regresores[i].predict(test)

        # Quedarse con la columna(clase) con la prediccion mas alta
        predicciones = np.argmax(salidas, axis = 1)

        return predicciones

# Apartado e) metodo predict_proba devuelve el array de probabilidadees filtradas por soft_max

    def predict_proba(self, test):

        n_muestras = test.shape[0]

        salidas = np.zeros((n_muestras, self.n_clases))

        for i in range(self.n_clases):

            salidas[:, i] = self.regresores[i].predict(test)

        # Ahora aplicar soft_max: soft_max(x1,x2,x3...) = 1/sumatorio(j=1 hasta n) de e^Xj (e^x1, e^x2, e^x3...)

        exponencial = np.exp(salidas)
        sumatorio = np.sum(exponencial, axis = 1, keepdims = True)
        probabilidades = exponencial / sumatorio

        return probabilidades

# Apartado f) metodo score recibe entradas y salidas y calcula la tasa de aciertos

    def score(self, X, y):

        # Convertir entrada en array por si acaso
        y_correcto = np.array(y).flatten()

        # Predecir
        y_predicha = self.predict(X)

        # Calcular la tasa
        aciertos = 0

        for i in range(len(y_correcto)):

            if(y_predicha[i] == y_correcto[i]):

                aciertos += 1

        n_muestras = len(y_correcto)
        tasa_aciertos = aciertos/ n_muestras

        return tasa_aciertos
        

Apartado g) aplicar HoldOut para probar la clase

In [42]:
#importar el repositiorio y escalar la X

dataset = fetch_ucirepo(id = 80)

X = dataset.data.features
Y = dataset.data.targets


X_escalado = X / 16.0

# Hallar numero de clases unicas
numero_clases = len(np.unique(Y.values.flatten()))

#Aplicar HoldOut 2/3-1/3, usar semilla definida para siempre mismo resultado
X_train, X_test, Y_train, Y_test = train_test_split(X_escalado, Y, test_size= 1/3, random_state = 42)


#Pasar al objeto los datos para probarlo
rlm_creado = rlm(n_clases = numero_clases)

rlm_creado.fit(X_train, Y_train)

tasa_aciertos = rlm_creado.score(X_test, Y_test)

In [43]:
tasa_aciertos

0.9306296691568837

# --------------------------------- EJERCICIO 2--------------------------

In [44]:
#importar el repositiorio y escalar la X

dataset = fetch_ucirepo(id = 80)

X = dataset.data.features
Y = dataset.data.targets


X_escalado = X / 16.0

X_array = np.array(X_escalado)
Y_array = np.array(Y).flatten()

numero_clases = len(np.unique(Y_array))

#Aplicar la validacion cruzada
validacion_cruzada = KFold(n_splits = 10, shuffle = True, random_state = 42)

resultados = []

for carpeta, (indices_train, indice_test) in enumerate(validacion_cruzada.split(X_array)):

    #Separar los datos usados para entrenar de los usados para test
    X_train, X_test = X_array[indices_train], X_array[indice_test]
    Y_train, Y_test = Y_array[indices_train], Y_array[indice_test]

    #Probar con rlm

    modelo_rlm = rlm(n_clases=numero_clases)
    modelo_rlm.fit(X_train, Y_train)
    aciertos_rlm = modelo_rlm.score(X_test, Y_test)

    # Probar con el de Sklearn

    regresion_logistica = LogisticRegression(max_iter = 1000)
    regresion_logistica.fit(X_train, Y_train)
    aciertos_regresion = regresion_logistica.score(X_test, Y_test)

    #Guardar resultados

    resultados.append({
        'TEST Carpeta': carpeta,
        'Aciertos RLM': aciertos_rlm,
        'Aciertos Rlog': aciertos_regresion
    })

# Crear y mostrar tabla

df_resultados = pd.DataFrame(resultados)
df_resultados.set_index('TEST Carpeta', inplace = True)

display(df_resultados)
    
    

,Aciertos RLM,Aciertos Rlog
TEST Carpeta,,
0,0.943060,0.964413
1,0.927046,0.966192
2,0.909253,0.955516
3,0.944840,0.976868
4,0.928826,0.973310
5,0.943060,0.976868
6,0.943060,0.971530
7,0.935943,0.971530
8,0.934164,0.964413


### Media total de cada uno

In [45]:
media_rlm = df_resultados['Aciertos RLM'].mean()

In [46]:
media_rlm

np.float64(0.9350533807829182)

In [47]:
media_rlog = df_resultados['Aciertos Rlog'].mean()

In [48]:
media_rlog

np.float64(0.9701067615658363)